# VideoSeek - Kiến Trúc 2-Agent (Actor Agent + Reflexion Evaluator Agent) Với Unified Tool (`inspect`)
Notebook này dùng để kiểm thử trên **Google Colab** cho kiến trúc **Dual-Agent Reflexion & Unified Multi-Granular Tool (`inspect`)**:
1. **Cell 1:** Clone/Cập nhật repository `videoseek` và cài đặt đầy đủ thư viện (`decord`, `litellm`, `mcp`, `pillow`, `pyyaml`, `yt-dlp`, `openai-whisper`, `opencv-python-headless`, `pytest`).
2. **Cell 2:** Thiết lập biến môi trường kết nối LLM/VLM API (`OPENAI_API_KEY`, `OPENAI_API_BASE`, `OPENAI_API_VERSION`, `MODEL_NAME = "gpt-6.1-terra"`).
3. **Cell 3:** Chạy bộ **Unit Tests** tự động (`tests/test_unified_reflexion.py`) kiểm tra `inspect_tool`, Macro Grid Mode ($N > 16$), Micro High-Res Mode ($N \le 16$), Frame Budget Guardrail, và sự tách biệt của `ReflexionEvaluatorAgent`.
4. **Cell 4:** Tạo **Synthetic Video** kiểm thử bằng OpenCV (video dài 120s có chèn dòng chữ màu đỏ `"BI MAT"` xuất hiện chớp nhoáng trong 3 giây).
5. **Cell 5:** Khởi chạy `VideoSeekAgent` với công cụ hợp nhất `inspect` + `ReflexionEvaluatorAgent`, hiển thị chi tiết `Thought -> Action -> Observation -> Evaluator Reflexion`.
6. **Cell 6:** Tự Upload Video của bạn (hoặc Đường dẫn file / Link YouTube) để kiểm thử với kiến trúc mới.
7. **Cell 7:** **A/B Benchmark** so sánh trực diện giữa **Baseline (VideoSeek gốc: 3 tools `overview`, `skim`, `focus`, không Reflexion)** và **Upgraded (Unified `inspect` + Reflexion Evaluator)**.

## Cell 1: Clone Repository & Cài Đặt Thư Viện Trên Google Colab

In [ ]:
import os
import sys

# 1. Clone repo nếu đang chạy trên Google Colab và chưa có thư mục videoseek
if not os.path.exists("videoseek") and not os.path.exists("videoseek/agent.py"):
    !git clone https://github.com/nhatsonle/videoseek.git

# Di chuyển vào thư mục gốc của repo videoseek
if os.path.exists("videoseek") and os.path.isdir("videoseek") and os.path.exists("videoseek/config"):
    os.chdir("videoseek")

!git pull origin main

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Current working directory:", os.getcwd())

# 2. Cài đặt đầy đủ các thư viện yêu cầu (bao gồm cả mcp cho litellm tool calling)
!pip install -q decord litellm mcp pillow pyyaml yt-dlp openai-whisper opencv-python-headless pytest

## Cell 2: Cấu Hình Biến Môi Trường API (`OPENAI_API_KEY`, `OPENAI_API_BASE`, `OPENAI_API_VERSION`)

In [ ]:
import os
import sys
import importlib

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith(("videoseek", "config")):
        importlib.reload(sys.modules[mod_name])

from config import general_config, prompts_config

# =====================================================================
# CẤU HÌNH API & MODEL TẠI ĐÂY:
# Cách 1: Điền trực tiếp API_KEY (sk-...) bên dưới
# Cách 2: Thêm 'OPENAI_API_KEY' vào mục Secrets (biểu tượng chìa khóa 🔑 bên trái Google Colab)
# =====================================================================
API_KEY = "YOUR_OPENAI_API_KEY"  # <-- Thay bằng key 'sk-...' của bạn
API_BASE = "https://hub.smilesoftware.org/v1"
API_VERSION = ""
MODEL_NAME = "gpt-6.1-terra"

def _resolve_config_value(secret_name: str, direct_value: str, placeholder: str = "YOUR_OPENAI_API_KEY") -> str:
    if direct_value and direct_value != placeholder:
        return direct_value
    try:
        from google.colab import userdata
        secret_val = userdata.get(secret_name)
        if secret_val and secret_val != placeholder:
            return secret_val
    except Exception:
        pass
    env_val = os.getenv(secret_name)
    if env_val and env_val != placeholder:
        return env_val
    return direct_value

os.environ["OPENAI_API_KEY"] = _resolve_config_value("OPENAI_API_KEY", API_KEY)
os.environ["OPENAI_API_BASE"] = _resolve_config_value("OPENAI_API_BASE", API_BASE)
os.environ["OPENAI_API_VERSION"] = _resolve_config_value("OPENAI_API_VERSION", API_VERSION)

# Chuẩn bị runtime config cho VideoSeekAgent + ReflexionEvaluatorAgent
runtime_config = {**general_config, **prompts_config}
runtime_config["api_key"] = os.environ["OPENAI_API_KEY"]
runtime_config["api_base"] = os.environ["OPENAI_API_BASE"]
runtime_config["api_version"] = os.environ["OPENAI_API_VERSION"] or None
runtime_config["model_name"] = _resolve_config_value("VIDEOSEEK_MODEL", MODEL_NAME)

masked_key = runtime_config["api_key"][:6] + "..." + runtime_config["api_key"][-4:] if len(runtime_config["api_key"]) > 10 else runtime_config["api_key"]
print("Configured Model:", runtime_config["model_name"])
print("Configured API Base:", runtime_config["api_base"])
print("Configured API Key:", masked_key)
if runtime_config["api_key"] == "YOUR_OPENAI_API_KEY" or not runtime_config["api_key"].startswith("sk-"):
    print("⚠️ CẢNH BÁO: Bạn chưa thay API_KEY hợp lệ (bắt đầu bằng 'sk-') ở Cell 2!")
print("Default Tools:", runtime_config.get("tools"))
print("Reflexion Enabled:", runtime_config.get("reflexion_enabled"))
print("Max Inspect Frames:", runtime_config.get("max_inspect_frames"))

## Cell 3: Chạy Bộ Unit Tests Tự Động Kiểm Tra Kiến Trúc `inspect` + Dual-Agent Reflexion (`unittest`)
Kiểm tra toàn diện 6 hạng mục trong `tests/test_unified_reflexion.py`:
1. Schema chuẩn của `inspect_tool` (`query`, `start_time`, `end_time`, `fps`).
2. `execute_inspect` ở **Macro Grid Mode ($N > 16$)**: Ghép mỗi 8 frames thành lưới $2 \times 4$ kèm `Timestamp Matrix`.
3. `execute_inspect` ở **Micro High-Res Mode ($N \le 16$)**: Gửi từng ảnh đơn độ nét cao kèm timestamp chính xác tới phần trăm giây (`:.2f`).
4. **Frame Budget Guardrail**: Tự động clamp khi `(end_time - start_time) * fps > max_inspect_frames`.
5. Sự tách biệt hoàn toàn giữa System Prompt và message context của `VideoSeekAgent` (Actor) và `ReflexionEvaluatorAgent` (Evaluator).
6. Luồng chạy End-to-End `Thought -> Action -> Observation -> Evaluator Reflexion`.

In [ ]:
import os
import sys
import importlib
import importlib.util
import unittest

if not os.path.exists("tests/test_unified_reflexion.py"):
    if os.path.exists("videoseek") and os.path.isdir("videoseek"):
        os.chdir("videoseek")
    !git pull origin main

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

for _pkg in ("decord", "litellm"):
    if _pkg in sys.modules and not getattr(sys.modules[_pkg], "__file__", None):
        del sys.modules[_pkg]

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith(("videoseek", "config")):
        importlib.reload(sys.modules[mod_name])

test_file_path = os.path.abspath("tests/test_unified_reflexion.py")
spec = importlib.util.spec_from_file_location("test_unified_reflexion", test_file_path)
test_mod = importlib.util.module_from_spec(spec)
sys.modules["test_unified_reflexion"] = test_mod
spec.loader.exec_module(test_mod)

suite = unittest.TestLoader().loadTestsFromTestCase(test_mod.TestUnifiedInspectAndReflexion)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful(), "Unit tests failed!"
print(f"\n[PASS] Toàn bộ {result.testsRun}/{result.testsRun} Unit Tests cho Unified Inspect & Dual-Agent Reflexion đã vượt qua thành công!")

## Cell 4: Tạo Video Kiểm Thử Giả Lập (Synthetic Video Bằng OpenCV)
Tạo một video kiểm thử (mặc định `120s`) gồm các cảnh chuyển động thông thường, nhưng tại khoảng **`44.0s - 47.0s`** (3 giây) xuất hiện đột ngột dòng chữ màu đỏ nổi bật **`"BI MAT"`** không có trong phụ đề `.srt`.

In [ ]:
import cv2
import numpy as np

SYNTHETIC_VIDEO_PATH = "synthetic_anomaly_test.mp4"
SYNTHETIC_SRT_PATH = "synthetic_anomaly_test.srt"
SYNTHETIC_DURATION_SEC = 120
SYNTHETIC_FPS = 10
ANOMALY_START_SEC = 44.0
ANOMALY_END_SEC = 47.0


def create_synthetic_test_video(
    video_path: str = SYNTHETIC_VIDEO_PATH,
    srt_path: str = SYNTHETIC_SRT_PATH,
    duration_sec: int = SYNTHETIC_DURATION_SEC,
    fps: int = SYNTHETIC_FPS,
    anomaly_start: float = ANOMALY_START_SEC,
    anomaly_end: float = ANOMALY_END_SEC,
):
    width, height = 640, 360
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(video_path, fourcc, fps, (width, height))
    total_frames = int(duration_sec * fps)

    for idx in range(total_frames):
        t = idx / float(fps)
        frame = np.full((height, width, 3), (45, 55, 65), dtype=np.uint8)
        cx = int((t / duration_sec) * (width - 100)) + 50
        cy = height // 2
        cv2.circle(frame, (cx, cy), 30, (200, 200, 200), -1)
        cv2.putText(
            frame,
            f"Time: {t:.1f}s",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (180, 180, 180),
            2,
        )

        if anomaly_start <= t <= anomaly_end:
            cv2.rectangle(frame, (80, 100), (560, 260), (0, 0, 40), -1)
            cv2.rectangle(frame, (80, 100), (560, 260), (0, 0, 255), 6)
            cv2.putText(
                frame,
                "BI MAT",
                (155, 205),
                cv2.FONT_HERSHEY_SIMPLEX,
                3.0,
                (0, 0, 255),
                8,
            )

        writer.write(frame)

    writer.release()

    srt_content = (
        "1\n"
        "00:00:00,000 --> 00:00:30,000\n"
        "Phần mở đầu giới thiệu chuyển động cơ bản của vật thể trên màn hình.\n\n"
        "2\n"
        "00:00:30,000 --> 00:01:30,000\n"
        "Vật thể hình tròn tiếp tục di chuyển đều từ trái sang phải.\n\n"
        "3\n"
        "00:01:30,000 --> 00:02:00,000\n"
        "Kết thúc quá trình quan sát chuyển động đều.\n"
    )
    with open(srt_path, "w", encoding="utf-8") as f:
        f.write(srt_content)

    print(f"Đã tạo video kiểm thử: {video_path} ({duration_sec}s @ {fps} FPS)")
    print(f"Sự kiện chữ đỏ 'BI MAT' được chèn tại: {anomaly_start}s - {anomaly_end}s")


create_synthetic_test_video()

## Cell 5: Khởi Chạy `VideoSeekAgent` Với Unified Tool (`inspect`) & `ReflexionEvaluatorAgent`
Chạy `VideoSeekAgent` (`tools=["inspect"]`, `reflexion_enabled=True`) trên video kiểm thử để quan sát quá trình phối hợp giữa **Actor Agent** và **Reflexion Evaluator Agent**.

In [ ]:
try:
    import mcp
except ImportError:
    !pip install -q mcp
!git pull origin main
import importlib, sys
for _pkg in ("decord", "litellm"):
    if _pkg in sys.modules and not getattr(sys.modules[_pkg], "__file__", None):
        del sys.modules[_pkg]
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith(("videoseek", "config")):
        importlib.reload(sys.modules[mod_name])

from config import general_config, prompts_config
from videoseek.agent import VideoSeekAgent

runtime_config.update(prompts_config)
for k, v in general_config.items():
    runtime_config.setdefault(k, v)
runtime_config["reflexion_enabled"] = True

test_question = (
    "Trong video có một dòng chữ in hoa màu đỏ xuất hiện chớp nhoáng trong khoảng 3 giây "
    "(không nằm trong phụ đề). Nội dung chính xác của dòng chữ màu đỏ đó là gì?"
)

agent = VideoSeekAgent(
    config=runtime_config,
    video_path=SYNTHETIC_VIDEO_PATH,
    subtitle_path=SYNTHETIC_SRT_PATH,
    output_dir="./outputs",
    tools=["inspect"],
    verbose=True,
)

trajectory = agent.run(test_question)

print("\n================ KẾT QUẢ KIỂM CHỨNG DUAL-AGENT REFLEXION ================")
print(f"1. Tổng số bước thực hiện: {len(trajectory.steps)}")
for step in trajectory.steps:
    fn_name = step.action.function_name
    params = step.action.parameters
    print(f"   - [Step {step.step_id}] Action: {fn_name} | Params: {params}")
    if step.reflection:
        first_line = step.reflection.strip().splitlines()[0]
        print(f"     ↳ Evaluator: {first_line}")

print(f"\n2. Final Answer: {trajectory.final_answer}")
print(f"3. Finish Reason: {trajectory.finish_reason}")

## Cell 6: Tự Upload Video Của Bạn (hoặc Đường Dẫn File / Link YouTube) & Chạy Kiểm Thử
Bạn có thể cung cấp video theo **1 trong 3 cách**:
1. **Upload trực tiếp từ máy tính:** Để `CUSTOM_VIDEO_PATH = ""`, khi chạy cell sẽ tự động bật nút **Choose Files** của Google Colab.
2. **Kéo thả file vào thanh Files bên trái Colab (hoặc Google Drive):** Điền đường dẫn vào `CUSTOM_VIDEO_PATH = "/content/ten_video.mp4"`.
3. **Dán trực tiếp Link YouTube:** Điền URL YouTube vào `CUSTOM_VIDEO_PATH = "https://www.youtube.com/watch?v=..."`.

In [ ]:
import os
import sys
import importlib
import json
import subprocess
from pathlib import Path

for _pkg in ("decord", "litellm"):
    if _pkg in sys.modules and not getattr(sys.modules[_pkg], "__file__", None):
        del sys.modules[_pkg]
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith(("videoseek", "config")):
        importlib.reload(sys.modules[mod_name])

from videoseek.agent import VideoSeekAgent

CUSTOM_VIDEO_PATH = ""  # Ví dụ: "/content/video_test.mp4" hoặc link YouTube
CUSTOM_SRT_PATH = ""    # Tuỳ chọn: Đường dẫn file phụ đề .srt (để trống sẽ tự tìm file .srt cùng tên)
CUSTOM_QUESTION = "Đồ vật gì được giấu đằng sau bức chân dung Hitler treo trên tường?"

def prepare_custom_video(video_input: str, srt_input: str = ""):
    video_path = video_input.strip()
    srt_path = srt_input.strip() if srt_input else None

    if not video_path:
        try:
            from google.colab import files
            print("📂 Vui lòng chọn file video (.mp4, .mkv, .mov, .avi) và file .srt (nếu có) từ máy tính của bạn:")
            uploaded = files.upload()
            for fname in uploaded.keys():
                abs_path = os.path.abspath(fname)
                if fname.lower().endswith(".srt"):
                    srt_path = abs_path
                elif fname.lower().endswith((".mp4", ".mkv", ".mov", ".avi", ".webm")):
                    video_path = abs_path
            if not video_path and uploaded:
                video_path = os.path.abspath(next(iter(uploaded.keys())))
        except Exception as e:
            raise RuntimeError(
                f"Không thể mở hộp thoại upload tự động ({e}). "
                "Hãy kéo thả video vào mục Files của Colab và điền đường dẫn vào biến CUSTOM_VIDEO_PATH."
            )

    elif video_path.startswith(("http://", "https://")) and ("youtube.com" in video_path or "youtu.be" in video_path):
        dl_dir = Path("./custom_downloads")
        dl_dir.mkdir(parents=True, exist_ok=True)
        output_template = str(dl_dir / "%(id)s.%(ext)s")
        print(f"🌐 Đang tải video từ YouTube: {video_path} ...")
        cmd = [
            "yt-dlp",
            "-f", "bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[height<=720][ext=mp4]/best",
            "--merge-output-format", "mp4",
            "-o", output_template,
            video_path,
        ]
        subprocess.run(cmd, check=True)
        mp4_files = sorted(dl_dir.glob("*.mp4"), key=os.path.getmtime, reverse=True)
        if not mp4_files:
            raise FileNotFoundError("Không tìm thấy file .mp4 sau khi tải bằng yt-dlp.")
        video_path = str(mp4_files[0].resolve())

    if not video_path or not os.path.exists(video_path):
        raise FileNotFoundError(f"Không tìm thấy file video tại: '{video_path}'")

    if not srt_path or not os.path.exists(srt_path):
        candidate_srt = str(Path(video_path).with_suffix(".srt"))
        if os.path.exists(candidate_srt):
            srt_path = candidate_srt
        else:
            srt_path = None

    return video_path, srt_path

resolved_video_path, resolved_srt_path = prepare_custom_video(CUSTOM_VIDEO_PATH, CUSTOM_SRT_PATH)
print(f"✅ Video sử dụng: {resolved_video_path}")
print(f"✅ Phụ đề (.srt): {resolved_srt_path or 'Không có (chỉ dùng thị giác)'}")

custom_agent = VideoSeekAgent(
    config=runtime_config,
    video_path=resolved_video_path,
    subtitle_path=resolved_srt_path,
    output_dir="./outputs",
    tools=["inspect"],
    verbose=True,
)

custom_trajectory = custom_agent.run(CUSTOM_QUESTION)

print("\n================ KẾT QUẢ CHẠY TRÊN VIDEO CỦA BẠN ================")
print(f"1. Tổng số bước thực hiện: {len(custom_trajectory.steps)}")
for step in custom_trajectory.steps:
    print(f"   - [Step {step.step_id}] Tool: {step.action.function_name} | Params: {step.action.parameters}")

print(f"\n2. Câu trả lời cuối cùng (Final Answer):\n{custom_trajectory.final_answer}")
print(f"3. Finish Reason: {custom_trajectory.finish_reason}")

os.makedirs("./outputs", exist_ok=True)
with open("./outputs/custom_trajectory.json", "w", encoding="utf-8") as f:
    json.dump(custom_trajectory.to_dict(), f, ensure_ascii=False, indent=2)
print("💾 Đã lưu chi tiết quỹ đạo vào: ./outputs/custom_trajectory.json")

## Cell 7: A/B Benchmark So Sánh Hiệu Năng Trước & Sau Khi Nâng Cấp (Cùng 1 Video & 1 Câu Hỏi)
Cell này tự động thực thi **2 lượt chạy đối đầu trực tiếp** trên cùng một Video và cùng một Câu hỏi:
1. **Lượt 1 - Baseline (VideoSeek gốc):** Sử dụng bộ 3 công cụ rời rạc `tools=["overview", "skim", "focus"]`, `reflexion_enabled=False` (không có Evaluator phản tư, dùng `LEGACY_SYSTEM_PROMPT`).
2. **Lượt 2 - Upgraded (Unified `inspect` + Dual-Agent Reflexion):** Sử dụng công cụ hợp nhất `tools=["inspect"]`, `reflexion_enabled=True` (Actor Agent phối hợp cùng `ReflexionEvaluatorAgent` độc lập).

Sau khi chạy xong, hệ thống sẽ in ra **Bảng So Sánh Hiệu Năng Đối Đầu (Head-to-Head Comparison)**: đo lường số turns, thời gian xử lý (giây), chuỗi công cụ đã gọi, và độ chính xác của đáp án.

In [ ]:
import time
import copy
import json
import os
import importlib
import sys
from pathlib import Path

for _pkg in ("decord", "litellm"):
    if _pkg in sys.modules and not getattr(sys.modules[_pkg], "__file__", None):
        del sys.modules[_pkg]
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith(("videoseek", "config")):
        importlib.reload(sys.modules[mod_name])

from config import general_config, prompts_config
from videoseek.agent import VideoSeekAgent

runtime_config.update(prompts_config)
for k, v in general_config.items():
    runtime_config.setdefault(k, v)

# =====================================================================
# CẤU HÌNH BÀI KIỂM THỬ A/B BENCHMARK:
# - Mặc định dùng video kiểm thử 120s có chữ đỏ 3s (hoặc đổi thành video của bạn)
# - Nhập câu hỏi muốn thử nghiệm đối đầu
# =====================================================================
BENCH_VIDEO_PATH = SYNTHETIC_VIDEO_PATH  # Hoặc đường dẫn video tùy ý (vd: "/content/my_video.mp4")
BENCH_SRT_PATH = SYNTHETIC_SRT_PATH      # Hoặc đường dẫn file .srt (để trống "" để tự động tìm file .srt cùng tên)
BENCH_QUESTION = (
    "Trong video có một dòng chữ in hoa màu đỏ xuất hiện chớp nhoáng trong khoảng 3 giây "
    "(không nằm trong phụ đề). Nội dung chính xác của dòng chữ màu đỏ đó là gì?"
)

# Tự động tìm file phụ đề .srt cùng tên với BENCH_VIDEO_PATH nếu có
if not BENCH_SRT_PATH or not os.path.exists(str(BENCH_SRT_PATH)):
    candidate_srt = str(Path(BENCH_VIDEO_PATH).with_suffix(".srt"))
    if os.path.exists(candidate_srt):
        BENCH_SRT_PATH = candidate_srt
        print(f"📄 Tự động tìm thấy file phụ đề cùng tên: {BENCH_SRT_PATH}")
    else:
        BENCH_SRT_PATH = None

print("=======================================================================")
print("🏁 BẮT ĐẦU A/B BENCHMARK: SO SÁNH TRỰC DIỆN TRƯỚC VÀ SAU NÂNG CẤP")
print(f"📹 Video: {BENCH_VIDEO_PATH}")
print(f"📄 Phụ đề (.srt): {BENCH_SRT_PATH or 'Không có'}")
print(f"❓ Question: {BENCH_QUESTION}")
print("=======================================================================\n")

# ---------------------------------------------------------------------
# 1. CHẠY LƯỢT 1: BASELINE (TRƯỚC NÂNG CẤP - VIDEOSEEK GỐC 3 TOOLS)
# ---------------------------------------------------------------------
print("⏳ [1/2] ĐANG CHẠY BASELINE (VideoSeek gốc: 3 tools ['overview', 'skim', 'focus'], reflexion_enabled=False)...")
cfg_baseline = copy.deepcopy(runtime_config)
cfg_baseline["reflexion_enabled"] = False

agent_baseline = VideoSeekAgent(
    config=cfg_baseline,
    video_path=BENCH_VIDEO_PATH,
    subtitle_path=BENCH_SRT_PATH if BENCH_SRT_PATH else None,
    output_dir="./outputs/baseline",
    tools=["overview", "skim", "focus"],
    verbose=True,
)

t0_base = time.time()
traj_base = agent_baseline.run(BENCH_QUESTION)
dur_base = time.time() - t0_base
steps_base = len(traj_base.steps)
tools_base = " -> ".join([s.action.function_name for s in traj_base.steps]) if traj_base.steps else "None"
ans_base = traj_base.final_answer.strip() if traj_base.final_answer else "None"
print(f"   └── Xong Baseline: {steps_base} turns | Thời gian: {dur_base:.1f}s | Finish: {traj_base.finish_reason}")

# ---------------------------------------------------------------------
# 2. CHẠY LƯỢT 2: UPGRADED (SAU NÂNG CẤP - UNIFIED INSPECT + REFLEXION EVALUATOR)
# ---------------------------------------------------------------------
print("\n🚀 [2/2] ĐANG CHẠY UPGRADED (Unified Tool ['inspect'] + Dual-Agent Reflexion Evaluator, reflexion_enabled=True)...")
cfg_upgraded = copy.deepcopy(runtime_config)
cfg_upgraded["reflexion_enabled"] = True

agent_upgraded = VideoSeekAgent(
    config=cfg_upgraded,
    video_path=BENCH_VIDEO_PATH,
    subtitle_path=BENCH_SRT_PATH if BENCH_SRT_PATH else None,
    output_dir="./outputs/upgraded",
    tools=["inspect"],
    verbose=True,
)

t0_upg = time.time()
traj_upg = agent_upgraded.run(BENCH_QUESTION)
dur_upg = time.time() - t0_upg
steps_upg = len(traj_upg.steps)
tools_upg = " -> ".join(
    [
        f"{s.action.function_name}({s.action.parameters.get('start_time')}-{s.action.parameters.get('end_time')}s@{s.action.parameters.get('fps')}fps)"
        if s.action.function_name == "inspect"
        else s.action.function_name
        for s in traj_upg.steps
    ]
) if traj_upg.steps else "None"
ans_upg = traj_upg.final_answer.strip() if traj_upg.final_answer else "None"
print(f"   └── Xong Upgraded: {steps_upg} turns | Thời gian: {dur_upg:.1f}s | Finish: {traj_upg.finish_reason}")

# ---------------------------------------------------------------------
# 3. BẢNG SO SÁNH HIỆU NĂNG ĐỐI ĐẦU CHI TIẾT (HEAD-TO-HEAD REPORT)
# ---------------------------------------------------------------------
print("\n" + "=" * 85)
print("📊 BẢNG SO SÁNH HIỆU NĂNG ĐỐI ĐẦU (HEAD-TO-HEAD BENCHMARK)")
print("=" * 85)
header = f"{'Tiêu Chí Đánh Giá':<26} | {'BASELINE (3 Tools Gốc)':<26} | {'UPGRADED (Inspect + Reflexion)':<26}"
print(header)
print("-" * 85)
print(f"{'Kiến trúc Agent':<26} | {'Single-Agent (Không phản tư)':<26} | {'Dual-Agent (Actor + Evaluator)':<26}")
print(f"{'Bộ Công Cụ Thị Giác':<26} | {'overview, skim, focus':<26} | {'Unified inspect(t1, t2, fps)':<26}")
print(f"{'Đóng Gói Ảnh Thích Ứng':<26} | {'Cố định theo từng tool':<26} | {'Macro 2x4 Grid / Micro High-Res':<26}")
print(f"{'Số Turns Suy Luận':<26} | {f'{steps_base} turns':<26} | {f'{steps_upg} turns':<26}")
print(f"{'Thời Gian Thực Thi':<26} | {f'{dur_base:.1f} giây':<26} | {f'{dur_upg:.1f} giây':<26}")
print(f"{'Trạng Thái Dừng':<26} | {str(traj_base.finish_reason):<26} | {str(traj_upg.finish_reason):<26}")
print(f"{'Chuỗi Tool Đã Gọi':<26} | {tools_base:<26} | {tools_upg:<26}")
print("-" * 85)
short_base = (ans_base[:24] + "...") if len(ans_base) > 24 else ans_base
short_upg = (ans_upg[:24] + "...") if len(ans_upg) > 24 else ans_upg
print(f"{'Đáp Án Cuối Cùng':<26} | {short_base:<26} | {short_upg:<26}")
print("=" * 85)

# Lưu báo cáo vào file JSON
report_data = {
    "video": BENCH_VIDEO_PATH,
    "question": BENCH_QUESTION,
    "baseline": {
        "architecture": "Single-Agent (overview, skim, focus)",
        "reflexion_enabled": False,
        "turns": steps_base,
        "duration_sec": round(dur_base, 2),
        "finish_reason": traj_base.finish_reason,
        "tools_called": tools_base,
        "prediction": ans_base,
    },
    "upgraded": {
        "architecture": "Dual-Agent (Unified inspect + ReflexionEvaluatorAgent)",
        "reflexion_enabled": True,
        "turns": steps_upg,
        "duration_sec": round(dur_upg, 2),
        "finish_reason": traj_upg.finish_reason,
        "tools_called": tools_upg,
        "prediction": ans_upg,
    },
}
os.makedirs("./outputs", exist_ok=True)
with open("./outputs/ab_comparison_report.json", "w", encoding="utf-8") as f:
    json.dump(report_data, f, ensure_ascii=False, indent=2)
print("💾 Đã lưu toàn bộ số liệu so sánh vào: ./outputs/ab_comparison_report.json")